# Install Dependencies

In [ ]:
!apt-get install tesseract-ocr -y
!apt-get install poppler-utils -y

!pip install pytesseract pdf2image opencv-python pandas pillow

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


# Import Libraries

In [ ]:
import pytesseract
import cv2
import pandas as pd
import numpy as np
import re

from pdf2image import convert_from_path
from google.colab import drive
from PIL import Image

# Upload the PDF

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Set the root path directory
pdf_path = "/content/drive/MyDrive/Data Science Challenge/anonymised 1.pdf"

In [ ]:
# Convert PDF to Images

pages = convert_from_path(pdf_path, dpi=300)

print("Total Pages:", len(pages))

# Image Pre-processing (Important for OCR accuracy)

In [ ]:
def preprocess_image(img):

    img = np.array(img)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    blur = cv2.GaussianBlur(gray,(5,5),0)

    thresh = cv2.threshold(blur,0,255,
                           cv2.THRESH_BINARY+cv2.THRESH_OTSU)[1]

    return thresh

# OCR Text Extraction

In [ ]:
all_text = ""

for i,page in enumerate(pages):

    processed = preprocess_image(page)

    text = pytesseract.image_to_string(processed)

    print("Page",i+1,"processed")

    all_text += text + "\n"

In [ ]:
# View Extracted Text

print(all_text[:2000])

# Extract Information Using Regex

def extract_information(text):

    application_numbers = re.findall(r'Application\s*No[:\s]*([A-Za-z0-9\/\-]+)', text)

    applicant_names = re.findall(r'(?:Applicant|Owner|Company)[\s:]*([A-Za-z\s\.]+)', text)

    application_dates = re.findall(r'\d{1,2}\.\s*\d{1,2}\.\s*\d{2,4}|\d{1,2}/\d{1,2}/\d{2,4}', text)

    document_types = re.findall(r'PART\s*\d+|CHARGING|REGISTER', text)

    return {
        "Application Numbers": application_numbers,
        "Applicant Names": applicant_names,
        "Application Dates": application_dates,
        "Document Types": document_types
    }

In [ ]:
# Extract Structured Data
info = extract_information(all_text)

info["Number_of_Pages"] = len(pages)

df = pd.DataFrame([info])

df

In [ ]:

!pip install -q -U google-generativeai

In [ ]:

import google.generativeai as genai

In [ ]:

# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

genai.configure(api_key=GOOGLE_API_KEY)